In [0]:
# WeatherPulse - Serverless PySpark Test

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("WeatherPulse Databricks Environment")
print("-----------------------------------")
print("Spark Version:", spark.version)

# Create a sample DataFrame
data = [
    ("Delhi", 18.87),
    ("Mumbai", 27.94),
    ("Chennai", 26.45),
    ("Bengaluru", 23.99)
]

columns = ["city_name", "avg_temperature"]

df = spark.createDataFrame(data, columns)

print("\nPySpark DataFrame Created Successfully!")

df.show()

print("Total Records:", df.count())

WeatherPulse Databricks Environment
-----------------------------------
Spark Version: 4.2.0

PySpark DataFrame Created Successfully!
+---------+---------------+
|city_name|avg_temperature|
+---------+---------------+
|    Delhi|          18.87|
|   Mumbai|          27.94|
|  Chennai|          26.45|
|Bengaluru|          23.99|
+---------+---------------+

Total Records: 4


In [0]:
# Create a dedicated schema for WeatherPulse

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.weatherpulse
""")

print("WeatherPulse schema created successfully!")

spark.sql("SHOW SCHEMAS IN workspace").show()

WeatherPulse schema created successfully!
+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
|      weatherpulse|
+------------------+



In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.weatherpulse.raw_data
""")

print("WeatherPulse raw data volume created!")

spark.sql("""
SHOW VOLUMES IN workspace.weatherpulse
""").show()

WeatherPulse raw data volume created!
+------------+-----------+
|    database|volume_name|
+------------+-----------+
|weatherpulse|   raw_data|
+------------+-----------+



In [0]:
# WeatherPulse - Bronze Layer
# Unity Catalog compatible code

from pyspark.sql.functions import current_timestamp, col

RAW_PATH = "/Volumes/workspace/weatherpulse/raw_data/"

# Read all JSON files
bronze_df = (
    spark.read
    .option("multiLine", "true")
    .json(RAW_PATH)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp())
)

print("Bronze Layer - Raw Data Preview")

bronze_df.show(5, truncate=False)

print("Total Rows:", bronze_df.count())

print("Total Columns:", len(bronze_df.columns))

print("Bronze Columns:")
print(bronze_df.columns)

Bronze Layer - Raw Data Preview
+---------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Save Bronze Layer as Delta Table

BRONZE_TABLE = "workspace.weatherpulse.bronze_weather_raw"

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print("Bronze Delta Table created successfully!")

display(spark.sql("""
    SELECT COUNT(*) AS total_records
    FROM workspace.weatherpulse.bronze_weather_raw
"""))

Bronze Delta Table created successfully!


total_records
7


In [0]:
# WeatherPulse - Silver Layer
# Flatten hourly weather arrays into individual records

from pyspark.sql.functions import (
    col, explode, arrays_zip, current_timestamp
)

# Read Bronze Delta Table
bronze_df = spark.table(
    "workspace.weatherpulse.bronze_weather_raw"
)

# Combine and flatten hourly arrays
silver_df = (
    bronze_df
    .withColumn(
        "weather",
        explode(
            arrays_zip(
                col("hourly.time"),
                col("hourly.temperature_2m"),
                col("hourly.relative_humidity_2m"),
                col("hourly.precipitation"),
                col("hourly.wind_speed_10m")
            )
        )
    )
    .select(
        col("source_file"),
        col("weather.time").alias("forecast_time"),
        col("weather.temperature_2m").alias("temperature"),
        col("weather.relative_humidity_2m").alias("humidity"),
        col("weather.precipitation").alias("precipitation"),
        col("weather.wind_speed_10m").alias("wind_speed"),
        col("latitude"),
        col("longitude"),
        col("timezone"),
        col("ingested_at")
    )
)

print("Silver Layer Transformation Successful!")

print("Total hourly records:", silver_df.count())

display(silver_df.limit(10))

Silver Layer Transformation Successful!
Total hourly records: 1176


source_file,forecast_time,temperature,humidity,precipitation,wind_speed,latitude,longitude,timezone,ingested_at
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T00:00,20.2,72,0.0,8.2,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T01:00,19.8,73,0.0,8.0,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T02:00,19.6,74,0.0,8.1,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T03:00,19.7,71,0.0,8.0,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T04:00,19.7,71,0.0,9.0,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T05:00,19.6,70,0.0,7.7,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T06:00,19.0,73,0.0,3.1,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T07:00,19.8,70,0.0,2.2,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T08:00,22.0,60,0.0,3.6,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z
dbfs:/Volumes/workspace/weatherpulse/raw_data/bengaluru_20261115T192858Z.json,2026-11-16T09:00,23.7,51,0.0,7.2,12.970123,77.56364,Asia/Kolkata,2026-11-16T17:09:47.977Z


In [0]:
# WeatherPulse - Save Silver Delta Table

SILVER_TABLE = "workspace.weatherpulse.silver_weather"

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

print("Silver Delta Table created successfully!")

display(spark.sql("""
    SELECT COUNT(*) AS total_records
    FROM workspace.weatherpulse.silver_weather
"""))

Silver Delta Table created successfully!


total_records
1176


In [0]:
# WeatherPulse - Silver Layer Data Quality Checks

from pyspark.sql.functions import col, sum, when, count

silver = spark.table("workspace.weatherpulse.silver_weather")

# 1. Total records
print("Total Records:", silver.count())

# 2. Missing values
print("\nMissing Values:")

silver.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in [
        "forecast_time",
        "temperature",
        "humidity",
        "precipitation",
        "wind_speed"
    ]
]).show()

# 3. Duplicate records
duplicates = (
    silver.groupBy("source_file", "forecast_time")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate Groups:", duplicates.count())

# 4. Invalid weather values
invalid = silver.filter(
    (col("humidity") < 0) |
    (col("humidity") > 100) |
    (col("precipitation") < 0) |
    (col("wind_speed") < 0)
)

print("Invalid Weather Records:", invalid.count())

Total Records: 1176

Missing Values:
+-------------+-----------+--------+-------------+----------+
|forecast_time|temperature|humidity|precipitation|wind_speed|
+-------------+-----------+--------+-------------+----------+
|            0|          0|       0|            0|         0|
+-------------+-----------+--------+-------------+----------+

Duplicate Groups: 0
Invalid Weather Records: 0


In [0]:
# WeatherPulse - Gold Layer
# City-wise Weather Summary

from pyspark.sql.functions import (
    avg, min, max, sum, round, count, when, col
)

silver = spark.table("workspace.weatherpulse.silver_weather")

# Extract city name from source filename
silver_with_city = silver.withColumn(
    "city_name",
    when(
        col("source_file").contains("bengaluru"), "Bengaluru"
    ).when(
        col("source_file").contains("chandigarh"), "Chandigarh"
    ).when(
        col("source_file").contains("chennai"), "Chennai"
    ).when(
        col("source_file").contains("delhi"), "Delhi"
    ).when(
        col("source_file").contains("ludhiana"), "Ludhiana"
    ).when(
        col("source_file").contains("mumbai"), "Mumbai"
    ).when(
        col("source_file").contains("patiala"), "Patiala"
    )
)

gold_city_summary = (
    silver_with_city
    .groupBy("city_name")
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        round(min("temperature"), 2).alias("min_temperature"),
        round(max("temperature"), 2).alias("max_temperature"),
        round(avg("humidity"), 2).alias("avg_humidity"),
        round(sum("precipitation"), 2).alias("total_precipitation"),
        round(max("wind_speed"), 2).alias("max_wind_speed"),
        count("*").alias("total_readings")
    )
)

print("Gold City Summary Created!")
display(gold_city_summary.orderBy("city_name"))

Gold City Summary Created!


city_name,avg_temperature,min_temperature,max_temperature,avg_humidity,total_precipitation,max_wind_speed,total_readings
Bengaluru,23.99,17.5,30.1,54.13,0.6,15.7,168
Chandigarh,17.81,13.2,24.7,63.8,0.0,16.0,168
Chennai,26.45,23.4,31.1,79.48,40.4,29.0,168
Delhi,18.87,13.7,26.3,66.39,0.0,12.1,168
Ludhiana,18.65,12.2,26.6,62.81,0.0,11.8,168
Mumbai,27.94,22.8,34.5,51.74,0.0,16.1,168
Patiala,18.21,11.7,26.2,65.07,0.0,14.7,168


In [0]:
# Save Gold City Summary as Delta Table

GOLD_TABLE = "workspace.weatherpulse.gold_city_weather_summary"

(
    gold_city_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

print("Gold City Summary Delta Table created!")

display(spark.table(GOLD_TABLE))

Gold City Summary Delta Table created!


city_name,avg_temperature,min_temperature,max_temperature,avg_humidity,total_precipitation,max_wind_speed,total_readings
Delhi,18.87,13.7,26.3,66.39,0.0,12.1,168
Chennai,26.45,23.4,31.1,79.48,40.4,29.0,168
Ludhiana,18.65,12.2,26.6,62.81,0.0,11.8,168
Chandigarh,17.81,13.2,24.7,63.8,0.0,16.0,168
Mumbai,27.94,22.8,34.5,51.74,0.0,16.1,168
Bengaluru,23.99,17.5,30.1,54.13,0.6,15.7,168
Patiala,18.21,11.7,26.2,65.07,0.0,14.7,168


In [0]:
# WeatherPulse - Gold Daily Weather Trend

from pyspark.sql.functions import (
    to_date, avg, sum, max, round, count
)

silver = spark.table("workspace.weatherpulse.silver_weather")

gold_daily_trend = (
    silver
    .withColumn("full_date", to_date("forecast_time"))
    .groupBy("full_date")
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        round(avg("humidity"), 2).alias("avg_humidity"),
        round(sum("precipitation"), 2).alias("total_precipitation"),
        round(max("wind_speed"), 2).alias("max_wind_speed"),
        count("*").alias("total_readings")
    )
    .orderBy("full_date")
)

print("Gold Daily Weather Trend Created!")

display(gold_daily_trend)

Gold Daily Weather Trend Created!


full_date,avg_temperature,avg_humidity,total_precipitation,max_wind_speed,total_readings
2026-11-16,22.42,58.95,0.0,16.0,168
2026-11-17,22.07,56.37,0.0,18.9,168
2026-11-18,21.72,58.57,9.8,25.8,168
2026-11-19,21.37,63.29,12.9,29.0,168
2026-11-20,21.4,64.62,15.3,22.4,168
2026-11-21,21.43,68.98,1.2,15.4,168
2026-11-22,21.5,72.64,1.8,12.6,168


In [0]:
# Save Gold Daily Trend as Delta Table

DAILY_TABLE = "workspace.weatherpulse.gold_daily_weather_trend"

(
    gold_daily_trend.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DAILY_TABLE)
)

print("Gold Daily Trend Delta Table created successfully!")

display(spark.table(DAILY_TABLE))

Gold Daily Trend Delta Table created successfully!


full_date,avg_temperature,avg_humidity,total_precipitation,max_wind_speed,total_readings
2026-11-16,22.42,58.95,0.0,16.0,168
2026-11-17,22.07,56.37,0.0,18.9,168
2026-11-18,21.72,58.57,9.8,25.8,168
2026-11-19,21.37,63.29,12.9,29.0,168
2026-11-20,21.4,64.62,15.3,22.4,168
2026-11-21,21.43,68.98,1.2,15.4,168
2026-11-22,21.5,72.64,1.8,12.6,168


In [0]:
# WeatherPulse - Gold Hourly Weather Trend

from pyspark.sql.functions import (
    hour, avg, round, count
)

silver = spark.table("workspace.weatherpulse.silver_weather")

gold_hourly_trend = (
    silver
    .withColumn("hour", hour("forecast_time"))
    .groupBy("hour")
    .agg(
        round(avg("temperature"), 2).alias("avg_temperature"),
        round(avg("humidity"), 2).alias("avg_humidity"),
        round(avg("wind_speed"), 2).alias("avg_wind_speed"),
        count("*").alias("total_readings")
    )
    .orderBy("hour")
)

print("Gold Hourly Weather Trend Created!")

display(gold_hourly_trend)

Gold Hourly Weather Trend Created!


hour,avg_temperature,avg_humidity,avg_wind_speed,total_readings
0,19.33,71.82,6.76,49
1,18.88,73.73,6.49,49
2,18.44,75.2,6.59,49
3,18.05,76.16,6.66,49
4,17.7,76.88,6.49,49
5,17.56,77.0,6.42,49
6,17.62,76.51,6.33,49
7,18.26,74.24,6.37,49
8,19.92,68.31,6.37,49
9,22.06,60.82,6.08,49


In [0]:
# Save Gold Hourly Trend as Delta Table

HOURLY_TABLE = "workspace.weatherpulse.gold_hourly_weather_trend"

(
    gold_hourly_trend.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(HOURLY_TABLE)
)

print("Gold Hourly Trend Delta Table created successfully!")

display(spark.table(HOURLY_TABLE))

Gold Hourly Trend Delta Table created successfully!


hour,avg_temperature,avg_humidity,avg_wind_speed,total_readings
0,19.33,71.82,6.76,49
1,18.88,73.73,6.49,49
2,18.44,75.2,6.59,49
3,18.05,76.16,6.66,49
4,17.7,76.88,6.49,49
5,17.56,77.0,6.42,49
6,17.62,76.51,6.33,49
7,18.26,74.24,6.37,49
8,19.92,68.31,6.37,49
9,22.06,60.82,6.08,49


In [0]:
# WeatherPulse - Final Medallion Layer Verification

tables = [
    "bronze_weather_raw",
    "silver_weather",
    "gold_city_weather_summary",
    "gold_daily_weather_trend",
    "gold_hourly_weather_trend"
]

print("WEATHERPULSE - DELTA TABLE VERIFICATION")
print("-" * 50)

for table_name in tables:
    full_name = f"workspace.weatherpulse.{table_name}"

    df = spark.table(full_name)

    print(f"{table_name}: {df.count()} records")

    print("Columns:", len(df.columns))
    print("-" * 50)

print("All Medallion Layer tables verified!")

WEATHERPULSE - DELTA TABLE VERIFICATION
--------------------------------------------------
bronze_weather_raw: 7 records
Columns: 11
--------------------------------------------------
silver_weather: 1176 records
Columns: 10
--------------------------------------------------
gold_city_weather_summary: 7 records
Columns: 8
--------------------------------------------------
gold_daily_weather_trend: 7 records
Columns: 6
--------------------------------------------------
gold_hourly_weather_trend: 24 records
Columns: 5
--------------------------------------------------
All Medallion Layer tables verified!


In [0]:
# Check current catalog and schema

print("Current Catalog:")
spark.sql("SELECT current_catalog()").show()

print("Current Schema:")
spark.sql("SELECT current_schema()").show()

print("WeatherPulse Tables:")
spark.sql("""
    SHOW TABLES IN workspace.weatherpulse
""").show(truncate=False)

Current Catalog:
+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+

Current Schema:
+----------------+
|current_schema()|
+----------------+
|         default|
+----------------+

WeatherPulse Tables:
+------------+-------------------------+-----------+
|database    |tableName                |isTemporary|
+------------+-------------------------+-----------+
|weatherpulse|bronze_weather_raw       |false      |
|weatherpulse|gold_city_weather_summary|false      |
|weatherpulse|gold_daily_weather_trend |false      |
|weatherpulse|gold_hourly_weather_trend|false      |
|weatherpulse|silver_weather           |false      |
+------------+-------------------------+-----------+

